In [1]:
import sys; print(sys.executable) 

/home/cjen/mySageMaker/ML/customer_sentiment/.venv/bin/python


In [2]:
!mkdir -p data

import boto3

region = boto3.Session().region_name
s3 = boto3.client("s3", region_name=region)

s3.download_file("sagemaker-sample-files", "datasets/text/SST2/sst2.train", "data/train")
s3.download_file("sagemaker-sample-files", "datasets/text/SST2/sst2.test", "data/test")
print("Download complete")

Download complete


In [3]:
from sagemaker import Session, s3

session = Session()

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/cjen/.config/sagemaker/config.yaml


### Upload the data

In [4]:
import boto3

region = boto3.Session().region_name
account = boto3.client("sts").get_caller_identity()["Account"]
bucket = f"sagemaker-{region}-{account}"

inputs = s3.S3Uploader.upload("data", "s3://{}/mxnet-gluon-sentiment-example/data".format(bucket))

print("S3 location " + inputs)

S3 location s3://sagemaker-us-west-2-493644444178/mxnet-gluon-sentiment-example/data


### Implement the training function - Run a SageMaker training job

In [5]:
from sagemaker.mxnet import MXNet

role = "arn:aws:iam::493644444178:role/SageMakerLocalExecutionRole"

m = MXNet(
    "sentiment.py",
    role=role,
    instance_count=1,
    instance_type="ml.m4.xlarge",
    framework_version="1.8.0",
    py_version="py37",
    distribution={"parameter_server": {"enabled": True}},
    hyperparameters={
        "batch-size": 8,
        "epochs": 2,
        "learning-rate": 0.01,
        "embedding-size": 50,
        "log-interval": 1000,
    },
)

In [6]:
m.fit(inputs)

INFO:sagemaker.image_uris:image_uri is not presented, retrieving image_uri based on instance_type, framework etc.
INFO:sagemaker:Creating training-job with name: mxnet-training-2026-03-20-02-47-37-959


2026-03-20 02:47:41 Starting - Starting the training job...
2026-03-20 02:47:55 Starting - Preparing the instances for training...
2026-03-20 02:48:25 Downloading - Downloading input data...
2026-03-20 02:48:50 Downloading - Downloading the training image...
2026-03-20 02:49:30 Training - Training image download completed. Training in progress..2026-03-20 02:49:38,389 sagemaker-training-toolkit INFO     Imported framework sagemaker_mxnet_container.training
2026-03-20 02:49:38,391 sagemaker-training-toolkit INFO     No GPUs detected (normal if no gpus installed)
2026-03-20 02:49:38,403 sagemaker_mxnet_container.training INFO     MXNet training environment: {'SM_HOSTS': '["algo-1"]', 'SM_NETWORK_INTERFACE_NAME': 'eth0', 'SM_HPS': '{"batch-size":8,"embedding-size":50,"epochs":2,"learning-rate":0.01,"log-interval":1000}', 'SM_USER_ENTRY_POINT': 'sentiment.py', 'SM_FRAMEWORK_PARAMS': '{"sagemaker_parameter_server_enabled":true}', 'SM_RESOURCE_CONFIG': '{"current_group_name":"homogeneousClus

In [7]:
predictor = m.deploy(initial_instance_count=1, instance_type="ml.m4.xlarge")

INFO:sagemaker:Repacking model artifact (s3://sagemaker-us-west-2-493644444178/mxnet-training-2026-03-20-02-47-37-959/output/model.tar.gz), script artifact (s3://sagemaker-us-west-2-493644444178/mxnet-training-2026-03-20-02-47-37-959/source/sourcedir.tar.gz), and dependencies ([]) into single tar.gz file located at s3://sagemaker-us-west-2-493644444178/mxnet-training-2026-03-20-02-56-50-901/model.tar.gz. This may take some time depending on model size...
INFO:sagemaker:Creating model with name: mxnet-training-2026-03-20-02-56-50-901
INFO:sagemaker:Creating endpoint-config with name mxnet-training-2026-03-20-02-56-50-901
INFO:sagemaker:Creating endpoint with name mxnet-training-2026-03-20-02-56-50-901


-----!

In [8]:
data = [
    "this movie was extremely good .",
    "the plot was very boring .",
    "this film is so slick , superficial and trend-hoppy .",
    "i just could not watch it till the end .",
    "the movie was so enthralling !",
]

response = predictor.predict(data)
print(response)

[1, 0, 0, 0, 1]


### Cleanup

In [9]:
predictor.delete_endpoint()

INFO:sagemaker:Deleting endpoint configuration with name: mxnet-training-2026-03-20-02-56-50-901
INFO:sagemaker:Deleting endpoint with name: mxnet-training-2026-03-20-02-56-50-901
